In [3]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report

# Set image size
img_height, img_width = 224, 224

# Data generator configuration
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=20,
    width_shift_range=0.2,
    height_shift_range=0.2,
    horizontal_flip=True
)

# Training data generator
train_generator = train_datagen.flow_from_directory(
    '/Users/shirongwei/Desktop/Lung X-Ray Image/Train',
    target_size=(img_height, img_width),
    batch_size=32,
    class_mode='categorical'
)

# Build CNN model
model = Sequential([
    Conv2D(32, (3, 3), activation='relu', input_shape=(img_height, img_width, 3)),
    MaxPooling2D((2, 2)),
    Conv2D(64, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(128, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Conv2D(256, (3, 3), activation='relu'),
    MaxPooling2D((2, 2)),
    Flatten(),
    Dense(256, activation='relu'),
    Dropout(0.5),
    Dense(3, activation='softmax')
])

# Compile the model
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])

# Callbacks
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='loss', patience=5, restore_best_weights=True),
    tf.keras.callbacks.ModelCheckpoint('best_model.h5', save_best_only=True)
]

# Train the model
history = model.fit(
    train_generator,
    epochs=30,
    callbacks=callbacks
)

# Test data generator
test_datagen = ImageDataGenerator(rescale=1./255)
test_generator = test_datagen.flow_from_directory(
    '/Users/shirongwei/Desktop/Lung X-Ray Image/Test',
    target_size=(img_height, img_width),
    batch_size=32,
    class_mode='categorical',
    shuffle=False
)

# Evaluate the model
test_loss, test_acc = model.evaluate(test_generator)

# Predictions and F1 score calculation
y_true = test_generator.classes
y_pred = np.argmax(model.predict(test_generator), axis=-1)

# Print classification report
class_indices = list(test_generator.class_indices.keys())
report = classification_report(y_true, y_pred, target_names=class_indices, output_dict=True)

# Extract F1 scores
f1_scores = {
    class_name: report[class_name]['f1-score']
    for class_name in class_indices
}
macro_f1 = report['macro avg']['f1-score']

print("F1 Scores for each class:")
for class_name, f1 in f1_scores.items():
    print(f"{class_name}: {f1:.4f}")

print(f"Macro-F1 Score: {macro_f1:.4f}")


Found 3000 images belonging to 3 classes.
Epoch 1/30
94/94 [==============================] - 54s 565ms/step - loss: 0.8370 - accuracy: 0.6087
Epoch 2/30
94/94 [==============================] - 52s 553ms/step - loss: 0.6705 - accuracy: 0.7127
Epoch 3/30
94/94 [==============================] - 62s 660ms/step - loss: 0.5867 - accuracy: 0.7677
Epoch 4/30
94/94 [==============================] - 54s 572ms/step - loss: 0.5426 - accuracy: 0.7800
Epoch 5/30
94/94 [==============================] - 56s 598ms/step - loss: 0.5238 - accuracy: 0.7867
Epoch 6/30
94/94 [==============================] - 53s 559ms/step - loss: 0.4927 - accuracy: 0.8037
Epoch 7/30
94/94 [==============================] - 54s 572ms/step - loss: 0.4704 - accuracy: 0.8093
Epoch 8/30
94/94 [==============================] - 54s 573ms/step - loss: 0.4625 - accuracy: 0.8180
Epoch 9/30
94/94 [==============================] - 54s 570ms/step - loss: 0.4267 - accuracy: 0.8263
Epoch 10/30
94/94 [==============================